# T3.4 — Deterministic noise injection

Inspect the Stage D manifest, exact noise quotas, and the seed-42 manual audit sample. Production generation remains in `scripts/11_inject_noise.py`.

In [1]:
import hashlib
import importlib.metadata as metadata
import json
import sys
from pathlib import Path

from nl2sparql.dataset.noise.artifacts import noise_artifact_lock

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
STAGE_D = ROOT / "data/dataset/raw/synthetic-stage-d.jsonl"
MANIFEST = ROOT / "data/dataset/raw/noise-config.json"
print(
    {
        "python": sys.version.split()[0],
        "project": metadata.version("nl2sparql-blockchain-kg"),
        "click": metadata.version("click"),
        "pydantic": metadata.version("pydantic"),
        "seed": 42,
    }
)

{'python': '3.11.15', 'project': '0.1.0', 'click': '8.4.1', 'pydantic': '2.13.4', 'seed': 42}


In [2]:
manifest = None
stage_d_bytes = None
with noise_artifact_lock(STAGE_D, MANIFEST):
    if not MANIFEST.exists() or not STAGE_D.exists():
        status = "Stage D manifest is absent: live Stage C remains gated by OPENROUTER_API_KEY."
    else:
        manifest = json.loads(MANIFEST.read_text())
        stage_d_bytes = STAGE_D.read_bytes()
        if hashlib.sha256(stage_d_bytes).hexdigest() != manifest["output"]["sha256"]:
            raise RuntimeError("Stage D bytes do not match the manifest hash")
if manifest is None:
    print(status)
else:
    display(
        {
            "source": manifest["source"],
            "output": manifest["output"],
            "quotas": manifest["config"]["quotas"],
            "quality": manifest["quality"],
        }
    )

Stage D manifest is absent: live Stage C remains gated by OPENROUTER_API_KEY.


In [3]:
if manifest is not None and stage_d_bytes is not None:
    rows = {
        row["id"]: row for row in (json.loads(line) for line in stage_d_bytes.decode().splitlines())
    }
    audit = manifest["manual_audit"]
    display(
        [
            {
                "id": record_id,
                "noise_type": rows[record_id]["noise_type"],
                "original": rows[record_id]["nl_original"],
                "noisy": rows[record_id]["nl"],
            }
            for record_id in audit["record_ids"]
        ]
    )
    if audit["completed"]:
        ratio = audit["decipherable_count"] / len(audit["record_ids"])
        print(
            {
                "decipherability_ratio": ratio,
                "acceptance_threshold": 0.90,
                "passes": ratio >= 0.90,
            }
        )
    else:
        print("Manual audit is incomplete; no ratio is claimed.")
else:
    print("The 30-row manual audit starts after Stage D generation.")

The 30-row manual audit starts after Stage D generation.
